# Scenario Configuration Reference

Every simulation in FHS is driven by a single YAML file. This notebook explains every field,
the assumptions behind each one, and how they connect to the simulation output.

**Files described here:**
- `notebooks/config/blockchain.yaml` — blockchain case study scenario
- `notebooks/config/advanced-features.yaml` — 10-feature advanced portfolio scenario
- `notebooks/config/scenario_schema.json` — JSON Schema that validates both files

---

## How a YAML field becomes a simulation result

```
YAML field                     Simulation step                    Output
─────────────────────────────────────────────────────────────────────────────────
expected_users  ──────────────► base conversion count
conversion_rate ──┐
uncertainty     ──┼────────────► random rate per scenario ──────► business value
acceptance_model──┘                                                distribution
business_value_per_conversion ─► EUR scaling                      (Expected, Floor,
annual_growth_rate ────────────► year-over-year compounding        P95, CVaR)
likelihood_of_non_delivery ───► zero-out in non-delivery draws
dependency_cluster ────────────► shared component-risk draw
risk_model ────────────────────► L2 market + L3 global multiplier
annual_operating_cost ─────────► cost overlay (shared inflation)
development_cost / installment► P&L charge per year
discount_rate ─────────────────► NPV discounting across years
```

Each YAML field enters the pipeline at a specific step. The sections below explain the
assumption behind each input and what happens if you change it.

## 1 — Scenario-level fields

These fields sit at the top of the YAML file and apply to the entire scenario.

```yaml
scenario_id: blockchain
name: Blockchain Scenario
budget: 155000.0
seed: 42
scenarios: 100000
discount_rate: 0.08
cost_inflation_max: 0.25
```

| Field | Type | Default | What it controls |
|---|---|---|---|
| `scenario_id` | string | — (required) | Unique identifier; used to load the file via `load_scenario("blockchain")` |
| `name` | string | `"Unnamed Scenario"` | Human-readable display name |
| `description` | string | `null` | Optional free-text description |
| `budget` | float ≥ 0 | `100000.0` | Total investment budget in EUR — used by budget check and portfolio optimiser |
| `seed` | integer | `42` | Random seed — guarantees identical output on every run |
| `scenarios` | integer | `100000` | Monte Carlo draws per feature per year |
| `discount_rate` | float 0–1 | `0.10` | Annual discount rate for NPV (e.g. `0.08` = 8%) |
| `cost_inflation_max` | float 0–1 | `0.25` | Maximum annual operating cost inflation — see section 3 |

---

### Assumption: `seed` — reproducibility

The seed initialises the random number generator. Every time you run the notebook with the
same seed, you get **exactly the same simulation output**.

**Why this matters:**
- Two colleagues running the same notebook get the same numbers — no "my results look different".
- Changing the seed and re-running is a simple sensitivity check: if the conclusion changes, the
  sample size is too small.

**When to change it:** When you want to verify that results are stable across different random
draws — run with seed 42, then with seed 99, and compare Expected and the business value floor. Stable results
confirm the scenario count is large enough.

---

### Assumption: `scenarios` — sample size vs. accuracy

Each scenario is one independent simulation draw. More scenarios = smoother distribution,
more stable statistics, but slower run.

| `scenarios` | Use case | Business Value Floor stability |
|---|---|---|
| 1,000 | Quick exploration, prototyping | Noisy — can vary ±5–10% between runs |
| 10,000 | Development, fast iteration | Acceptable for most decisions |
| 100,000 | Presentation, board-level decisions | Stable — varies < 1% between runs |
| 1,000,000 | Audit-grade precision | Very stable, slow |

**Assumption:** 100,000 scenarios is the default because the business value floor and CVaR require at least
5,000 scenarios in the tail (5% of 100,000) to be statistically reliable.

---

### Assumption: `discount_rate` — time value of money

A euro received in year 3 is worth less than a euro today, because you could have invested
it in the meantime. The discount rate expresses that cost of waiting.

**NPV formula:**

```
NPV = Σ  (business_value_year_t) / (1 + discount_rate)^t  −  development_cost
```

**Typical values by context:**

| Context | Typical rate | Rationale |
|---|---|---|
| Risk-free (government bonds) | 2–4% | Opportunity cost of safe investment |
| Corporate hurdle rate | 8–12% | Minimum return required by the business |
| High-risk / startup | 15–25% | Higher uncertainty requires higher expected return |
| Blockchain scenario | 8% | Moderate corporate hurdle rate |

**Rule of thumb:** Use your organisation's official hurdle rate. If unsure, 10% is a
conservative default for corporate software investments.

## 2 — Feature-level fields

Each entry under `features:` describes one hypothesis. Required fields are marked ✅.

```yaml
features:
- name: 'H1: Simplified UI'
  expected_users: 100000
  conversion_rate: 0.26
  uncertainty: 0.4
  business_value_per_conversion: 4.7
  development_cost: 75000.0
  installment_years: 3
  annual_operating_cost: 25000.0
  development_weeks: 8
  annual_growth_rate: 0.05
  likelihood_of_non_delivery: 0.2
  dependency_cluster: Customer Experience Platform
  planned_release: R2
  acceptance_model: binomial
```

---

### 2a — Business value inputs

| Field | Type | Required | Default | What it means |
|---|---|:---:|---|---|
| `name` | string | ✅ | — | Feature identifier — must be unique within the scenario |
| `expected_users` | integer > 0 | ✅ | — | Users expected to see or use this feature |
| `conversion_rate` | float 0–1 | ✅ | — | Share of users expected to convert |
| `uncertainty` | float 0–1 | ✅ | — | Confidence in the conversion rate — see assumption below |
| `business_value_per_conversion` | float | — | `1.0` | EUR value per converted user |
| `annual_growth_rate` | float −1–5 | — | `0.0` | Year-over-year business value growth — see assumption below |
| `acceptance_model` | string | — | `"rate"` | `"rate"` or `"binomial"` — see assumption below |

---

### Assumption: business value formula

The deterministic (spreadsheet) answer before uncertainty is applied:

```
Expected Business Value = expected_users × conversion_rate × business_value_per_conversion
```

**H1 example:**

```
100,000 users × 26% conversion × €4.70/conversion = €122,200
```

The Monte Carlo simulation then varies the conversion rate according to `uncertainty`,
runs this formula for each scenario, and returns the distribution of results.

**What `business_value_per_conversion` represents** depends on your business model:

| Business model | What one conversion means | Typical unit |
|---|---|---|
| E-commerce feature | One purchase | Average order value (EUR) |
| Retention feature | One user not churning | Monthly subscription fee |
| Efficiency feature | One process automated | Cost saving per instance (EUR) |
| Internal tool | One task completed faster | Staff hour saved × hourly rate |

---

### Assumption: `uncertainty` — how confident are you in the conversion rate?

`uncertainty` is a relative measure of how much the actual conversion rate could deviate
from the configured `conversion_rate`.

```
uncertainty: 0.4   means the rate could realistically be ±40% of 0.26
             → range roughly  0.16 – 0.36  in most scenarios
```

**Distribution selection rule (automatic):**

| `uncertainty` | Distribution | Why |
|---|---|---|
| < 0.30 | Normal (bell curve) | Symmetric uncertainty — equally likely above and below |
| ≥ 0.30 | Lognormal (right-skewed) | High uncertainty — outcomes cannot go below zero but can surprise on the upside |

**Choosing a value — domain guidance:**

| Confidence level | `uncertainty` | Typical situation |
|---|---|---|
| High confidence | 0.05 – 0.15 | Mature feature, A/B test data available, similar features shipped before |
| Moderate | 0.20 – 0.35 | Some user research, analogues from other products |
| Low confidence | 0.40 – 0.60 | New market, no prior data, expert guess only |
| Very uncertain | > 0.60 | Exploratory hypothesis, early discovery stage |

**What happens when you change it:**
- Lower uncertainty → narrower distribution → Expected and the floor are closer together → more predictable outcome.
- Higher uncertainty → wider distribution → larger gap between Expected and the floor → riskier bet.

---

### Assumption: `acceptance_model` — how users convert

Two models are available:

| Model | Mechanism | When to use |
|---|---|---|
| `rate` | One aggregate conversion rate is drawn per scenario. All users contribute fractional conversions. | Legacy behaviour, continuous outputs, internal KPI features |
| `binomial` | Each user independently converts with probability = drawn rate. Output is always a whole-number count. | User-facing features, A/B tests, e-commerce, any case where "half a conversion" makes no sense |

**Example with H1 (100,000 users, 26% rate, uncertainty 40%):**

```
rate model:     conversion count = 100,000 × 0.247  = 24,700.0  (decimal)
binomial model: conversion count = Binomial(100,000, 0.247) = 24,683  (integer)
```

The binomial model produces more realistic counts and slightly wider distributions because it
adds the natural per-user variance on top of the rate uncertainty.

**Recommendation:** Use `binomial` for all user-facing features.

---

### Assumption: `annual_growth_rate` — year-over-year compounding

For multi-year simulations, the business value in each year grows by this rate relative to
the previous year. Compounding is applied **after** the uncertainty draw.

```
year_1_value = base_business_value  (from conversion simulation)
year_2_value = year_1_value × (1 + annual_growth_rate)
year_3_value = year_2_value × (1 + annual_growth_rate)
```

**H1 example (5% annual growth, expected year-1 value €122,200):**

| Year | Formula | Expected Value |
|---|---|---|
| Year 1 | base | €122,200 |
| Year 2 | × 1.05 | €128,310 |
| Year 3 | × 1.05 | €134,726 |
| 3-year total | | €385,236 |

**Guidance on setting this value:**

| Context | Rate | Rationale |
|---|---|---|
| Stable mature market | 0.0 – 0.05 | Little user base growth expected |
| Growing segment | 0.10 – 0.20 | Expanding into new regions or user groups |
| Platform / network effect | 0.20 – 0.40 | Viral adoption, strong retention loop |
| Declining product | −0.05 – −0.20 | Market saturation or competitive pressure |

**Note:** A negative rate means business value shrinks year over year — use this for features
in declining markets or those with known usage decay.

---

### 2b — Investment and cost inputs

| Field | Type | Required | Default | What it means |
|---|---|:---:|---|---|
| `development_cost` | float ≥ 0 | — | `0.0` | One-time build cost in EUR |
| `installment_years` | integer 1–30 | — | `1` | Straight-line installment period |
| `annual_operating_cost` | float ≥ 0 | — | `0.0` | Annual production cost (EUR/year) |
| `development_weeks` | integer ≥ 1 or null | — | `null` | Build duration — required for sprint planning |

---

### 2c — Risk and delivery inputs

| Field | Type | Required | Default | What it means |
|---|---|:---:|---|---|
| `likelihood_of_non_delivery` | float 0–1 | — | `0.0` | Probability the feature is never delivered |
| `dependency_cluster` | string or null | — | `null` | Shared platform/team for correlated risk |
| `planned_release` | string or null | — | `null` | Release label (e.g. `"R2"`) |

---

### Assumption: `likelihood_of_non_delivery` — what happens when a feature is not delivered?

In each simulation scenario, a uniform random draw decides whether this feature is delivered.
If the draw falls below `likelihood_of_non_delivery`, the feature's business value for that
scenario is set to **zero** — it contributes nothing to the portfolio.

```
likelihood_of_non_delivery: 0.20   → in 20% of scenarios, this feature delivers €0
                                       in 80% of scenarios, the full simulation runs
```

**Effect on statistics:**
- Expected value is reduced by the non-delivery probability: `Expected × (1 − non_delivery_risk)`
- Business value floor is pulled down because some scenarios are guaranteed zeros
- The feature still appears in the budget check (development cost is committed regardless)

**Typical values:**

| Situation | Value | Rationale |
|---|---|---|
| Committed, team ready | 0.05 – 0.10 | Small residual risk from unexpected blockers |
| Normal delivery risk | 0.15 – 0.25 | Dependency on other teams, unclear requirements |
| High delivery risk | 0.30 – 0.50 | New technology, external vendor dependency |
| Exploratory / speculative | 0.50 – 0.80 | Research spike, unproven approach |

---

### Assumption: `dependency_cluster` — correlated failure between features

When two or more features share the same `dependency_cluster`, they share a **single**
component-risk draw per scenario. If the shared platform fails in that scenario, **all**
features in the cluster are hit simultaneously.

**Why this matters — without clusters vs. with clusters:**

```
Without cluster (independent):
  Feature A fails independently with 8% probability
  Feature B fails independently with 8% probability
  → both fail in same scenario: 0.08 × 0.08 = 0.64% of scenarios

With shared cluster (correlated):
  "Customer Platform" cluster fails with 8% probability
  → if it fails, both A and B fail in that scenario: 8% of scenarios
```

Correlated failure is more realistic for features that share the same database,
API gateway, authentication service, or development team.

**Naming convention:** Use a descriptive platform or team name (e.g. `"Customer Platform"`,
`"Payment API"`, `"Team Alpha"`). The same string in `component_risk_by_cluster` sets the
failure probability for that cluster.

## 3 — Operating costs and cost inflation

Building a feature is a one-time investment. **Running it every year costs money too** — servers, cloud services, third-party APIs, monitoring.

### `annual_operating_cost` (per feature)

Set one value per feature. It represents the annual cost to keep that feature live.

```yaml
- name: 'H1: Simplified UI'
  annual_operating_cost: 25000.0   # app hosting, CDN, push notifications

- name: 'H2: Traceability'
  annual_operating_cost: 40000.0   # blockchain node, event streaming, telemetry storage

- name: 'H3: Expiration Alerts'
  annual_operating_cost: 3000.0    # alert delivery, minimal backend compute
```

**What belongs here:** Recurring costs that exist *because this feature is live* —
cloud compute, third-party API fees, storage, licences tied to this feature, and support.

**What does not belong here:** One-time development cost (`development_cost`),
shared platform costs that exist regardless of the feature, or team salaries
(those are in the delivery risk model under `delivery_risk`).

### `cost_inflation_max` (scenario level)

Operating costs are not perfectly predictable. Cloud prices change. Contracts get renegotiated. Vendor fees go up.

```yaml
cost_inflation_max: 0.25   # up to 25% annual increase
```

FHS models this with a **single shared inflation factor per simulation scenario**:

```
inflation_factor ~ Uniform(0, cost_inflation_max)
operating_cost = annual_operating_cost × (1 + inflation_factor)
```

**Why one shared factor?** All features run on the same infrastructure (same cloud, same hosting contract).
If infrastructure prices rise, they rise for all features at once — not independently.

| Inflation scenario | Factor | H1 cost | H2 cost | H3 cost | Portfolio total |
|---|---|---|---|---|---|
| Base (no inflation) | 0% | €25,000 | €40,000 | €3,000 | €68,000 |
| Expected (~12.5% avg) | 12.5% | €28,125 | €45,000 | €3,375 | €76,500 |
| Worst case (max 25%) | 25% | €31,250 | €50,000 | €3,750 | €85,000 |

## 4 — Installment

Development cost is a one-time spend, but accounting spreads it over multiple years.
FHS uses **straight-line installment**:

```
annual_installment = development_cost / installment_years
```

```yaml
- name: 'H1: Simplified UI'
  development_cost: 75000.0
  installment_years: 3       # → €25,000/year charged over 3 years

- name: 'H3: Expiration Alerts'
  development_cost: 20000.0
  installment_years: 1       # → full €20,000 charged in year 1 (default)
```

| `installment_years` | Effect |
|---|---|
| `1` (default) | Full development cost appears as a year-1 P&L charge |
| `3` | Cost spread equally: one-third per year for 3 years |
| `n` | Cost spread equally: `development_cost / n` per year |

**When to use more than 1 year:** When your accounting team capitalises the development spend
and amortises it — common for larger platform features with multi-year useful life.

**Blockchain scenario — installment overview:**

| Feature | Dev Cost | `installment_years` | Annual Charge |
|---|---|---|---|
| H1: Simplified UI | €75,000 | 3 | €25,000/yr |
| H2: Traceability | €50,000 | 2 | €25,000/yr |
| H3: Expiration Alerts | €20,000 | 2 | €10,000/yr |
| **Portfolio total** | **€145,000** | — | **€60,000/yr** |

## 5 — Risk model

The `risk_model` block configures three portfolio-wide risk layers that stack on top of the
per-feature uncertainty simulation.

```yaml
risk_model:
  risk_2_market_probability: 0.20
  risk_2_market_multiplier: 0.85
  risk_3_global_probability: 0.05
  risk_3_global_multiplier: 0.60
  component_risk_multiplier: 0.70
  default_component_probability: 0.08
  component_risk_by_cluster:
    Customer Platform: 0.08
    Supply Operations Platform: 0.12
```

| Field | What it models |
|---|---|
| `risk_2_market_probability` | Probability of a market downturn in a given scenario |
| `risk_2_market_multiplier` | If triggered: all feature business values × this factor |
| `risk_3_global_probability` | Probability of a severe global event (pandemic, regulation) |
| `risk_3_global_multiplier` | If triggered: all business values × this factor |
| `component_risk_multiplier` | Multiplier applied when a shared dependency/platform fails |
| `default_component_probability` | Default failure probability for clusters not listed explicitly |
| `component_risk_by_cluster` | Per-cluster failure probability overrides |

**Risk 1** (non-delivery) is configured per feature via `likelihood_of_non_delivery` — not here.

---

### Assumption: how the three risk layers stack in one scenario

All three layers are evaluated **independently** in every scenario. Their effects multiply.

```
Step 1 — L1: Non-delivery (per feature)
  Draw ∈ Uniform(0,1). If draw < likelihood_of_non_delivery → feature value = 0, stop.

Step 2 — L2: Market shock (portfolio-wide)
  Draw ∈ Uniform(0,1). If draw < risk_2_market_probability:
    all feature values ×= risk_2_market_multiplier   (e.g. × 0.85 = −15%)

Step 3 — L3: Global crisis (portfolio-wide)
  Draw ∈ Uniform(0,1). If draw < risk_3_global_probability:
    all feature values ×= risk_3_global_multiplier   (e.g. × 0.60 = −40%)

Step 4 — Component risk (per cluster)
  For each dependency_cluster: one draw ∈ Uniform(0,1).
  If draw < cluster_probability:
    all features in this cluster ×= component_risk_multiplier   (e.g. × 0.70 = −30%)
```

**Example — a single bad scenario where all layers fire:**

```
Base business value for H1:  €122,200

L1 (non-delivery):  draw = 0.12 < 0.20 → H1 delivers €0 in this scenario
                    (remaining steps skipped for H1)

L2 (market shock):  draw = 0.17 < 0.20 → all surviving features × 0.85
L3 (global crisis): draw = 0.03 < 0.05 → all surviving features × 0.60

Combined market + global multiplier on surviving features: 0.85 × 0.60 = 0.51 (−49%)
```

**Why independent draws?** Market shocks and global crises can co-occur. In 2020 a pandemic
(L3-type event) triggered simultaneous market shocks. The model allows both to fire in the
same scenario, which produces realistic extreme-tail scenarios.

**Calibrating the parameters:**

| Parameter | Conservative | Moderate | Stressed |
|---|---|---|---|
| `risk_2_market_probability` | 0.10 | 0.20 | 0.35 |
| `risk_2_market_multiplier` | 0.90 | 0.85 | 0.70 |
| `risk_3_global_probability` | 0.02 | 0.05 | 0.10 |
| `risk_3_global_multiplier` | 0.70 | 0.60 | 0.40 |

Use the "Stressed" column for board-level worst-case presentations. Use "Conservative" for
baseline planning.

## 6 — Delivery risk (sprint model)

Optional block. When present, FHS simulates sprint overruns and cancellation risk.
Leave this block out entirely if you do not need sprint-level delivery modelling.

```yaml
delivery_risk:
  sprint_length_weeks: 2
  quarterly_capacity_sprints: 6
  delay_model:
    sprint_uncertainty: 30
    sprint_ceiling: 3.0
  cancellation:
    max_sprints_over_plan: 4
    cancellation_probability: 0.5
```

| Field | Default | What it controls |
|---|---|---|
| `sprint_length_weeks` | 2 | Sprint length in weeks |
| `quarterly_capacity_sprints` | 6 | Available sprints per quarter |
| `sprint_uncertainty` | 30 | Sprint duration volatility in percent |
| `sprint_ceiling` | 3.0 | Hard cap — a sprint can be at most 3× its planned length |
| `max_sprints_over_plan` | 4 | Tolerance gate for delay: check starts only when `actual > planned + gate` (strictly `>`) |
| `cancellation_probability` | 0.5 | Historical probability of choosing abort (vs continue with another recovery sprint) once the gate is exceeded |

---

### Assumption: why truncated lognormal for sprint delays?

Sprint delays are modelled with a **truncated lognormal distribution**.

**Why truncated lognormal?**

- A sprint can never finish in negative time.
- Delays are asymmetric: large overruns are possible, large underruns are rare.
- Truncation adds a realistic hard ceiling (`sprint_ceiling`) without probability artifacts.

```
delay_factor ~ TruncatedLogNormal(lower=1.0, upper=sprint_ceiling)

Most likely outcome:   delay_factor ≈ 1.0  (on-time)
Occasional overrun:    delay_factor = 1.5  (50% longer)
Rare worst case:       delay_factor = 2.5  (bounded by sprint_ceiling = 3.0)
```

**What `sprint_uncertainty = 30` means:**
This is the relative variability of delivery speed in percent.
At 30, about two-thirds of sprints finish within ±30% of planned length.

| `sprint_uncertainty` | Team profile |
|---|---|
| 10 – 15 | Very predictable — mature team, stable scope |
| 20 – 30 | Normal — some scope creep, occasional surprises |
| 35 – 50 | Volatile — new team, unclear requirements, frequent replanning |

---

### Assumption: sprint cost calculation

Delivery cost is derived from feature data:

```
weekly_burn = development_cost / development_weeks
cost_per_sprint = weekly_burn × sprint_length_weeks
actual_cost = actual_sprints × cost_per_sprint
```

A delayed feature costs proportionally more because the team works for more sprints.

---

### Assumption: cancellation logic

Once a feature has overrun by more than `max_sprints_over_plan` sprints, cancellation
is evaluated:

```
if sprints_overrun > max_sprints_over_plan:
    draw ∈ Uniform(0,1)
    if draw < cancellation_probability:
        feature is cancelled → business value = 0, sunk cost is locked in
```

Domain interpretation of the decision gate:
- A started sprint always incurs full sprint cost.
- Delay/failure is recognized at sprint end (review boundary).
- `cancellation_probability` is calibrated from historical decision behavior
  (how often leadership aborts vs approves another recovery sprint after threshold breach).

In the blockchain scenario, this determines how often the model stops a delayed feature and
reports expected sunk cost as a delivery-risk metric in notebook 06.

## 7 — Strategy

The `strategy` block attaches a business category and rationale to each feature.
Used in portfolio advisor views and reporting.

```yaml
strategy:
  'H1: Simplified UI':
    category: Protect & Grow
    reason: Retain customers and grow share by fixing UX friction
  'H2: Traceability':
    category: Grow
    reason: Win new customers and premium pricing by proving sustainability
```

The key must exactly match the feature `name`. Category and reason are free text —
choose labels that match your organisation's strategic framework.

## 8 — How to add a new feature to a scenario

**Step 1 — Open the YAML file** for the scenario you want to extend:
`notebooks/config/blockchain.yaml`

**Step 2 — Add a new entry** under `features:`. Start with the four required fields,
then add optional fields as needed:

```yaml
- name: 'H4: Push Notifications'          # required — unique name
  expected_users: 80000                    # required — who will see this feature?
  conversion_rate: 0.12                    # required — what fraction will engage?
  uncertainty: 0.45                        # required — how confident are you? (0=certain, 1=guess)

  # Optional — business value
  business_value_per_conversion: 2.50      # EUR per engaged user
  annual_growth_rate: 0.08                 # 8% YoY growth

  # Optional — investment
  development_cost: 30000.0               # one-time build cost
  installment_years: 2                   # spread cost over 2 years
  development_weeks: 4                    # needed for sprint planning
  annual_operating_cost: 8000.0           # push service + cloud messaging

  # Optional — risk
  likelihood_of_non_delivery: 0.15        # 15% chance of not shipping
  dependency_cluster: Customer Platform   # shares risk with H1, H3
  planned_release: R3
  acceptance_model: binomial              # per-user yes/no — recommended
```

**Step 3 — Add a strategy entry** (optional but recommended for portfolio views):

```yaml
strategy:
  'H4: Push Notifications':
    category: Retention Engine
    reason: Re-engage inactive users with personalised alerts
```

**Step 4 — Reload the scenario** by rerunning the setup cell in notebook 02.
The configuration form and all simulations update automatically.

---

### Quick-reference: minimum viable feature entry

```yaml
- name: 'My Feature'
  expected_users: 10000
  conversion_rate: 0.15
  uncertainty: 0.30
```

All other fields default to zero / null. You can add them incrementally as you refine
your estimates.

In [ ]:
## 9 — Load and inspect the scenario live

In [ ]:
from fhs.notebook import load_scenario
from fhs.presentation.notebook import show

scenario = load_scenario("blockchain")

show.info(
    f"Scenario: {scenario.name} | "
    f"Budget: €{scenario.budget:,.0f} | "
    f"Discount rate: {scenario.discount_rate:.0%} | "
    f"Cost inflation max: {scenario.cost_inflation_max:.0%} | "
    f"Features: {len(scenario.features)}"
)

In [ ]:
rows = [
    (
        f.name,
        f"{f.expected_users:,}",
        f"{f.conversion_rate:.0%}",
        f"±{f.uncertainty:.0%}",
        f"€{f.business_value_per_conversion:,.2f}",
        f"€{f.development_cost:,.0f}",
        f"{f.installment_years}y",
        f"€{f.annual_operating_cost:,.0f}",
        f"{f.likelihood_of_non_delivery:.0%}",
    )
    for f in scenario.features
]

In [ ]:
show.samples(
    headers=(
        "Feature",
        "Users",
        "Conv. Rate",
        "Uncertainty",
        "Value/Conv",
        "Dev Cost",
        "Deprec.",
        "Oper. Cost/yr",
        "Non-Delivery",
    ),
    rows=rows,
    title="Blockchain scenario — all feature parameters",
)

### Live: uncertainty effect on the business value distribution

The cell below simulates H1 three times with different `uncertainty` values so you can
see how a single parameter change shifts the floor and spread of outcomes.

In [ ]:
from fhs import Feature
from fhs.application import BlockchainCaseStudyService

h1_base = scenario.features[0]  # H1: Simplified UI

uncertainty_levels = [0.10, 0.40, 0.65]
labels = [
    "Low (0.10) — Normal dist",
    "Medium (0.40) — Lognormal",
    "High (0.65) — Lognormal",
]

In [ ]:
rows = []
for u, label in zip(uncertainty_levels, labels, strict=False):
    f = Feature(
        name=h1_base.name,
        expected_users=h1_base.expected_users,
        conversion_rate=h1_base.conversion_rate,
        uncertainty=u,
        business_value_per_conversion=h1_base.business_value_per_conversion,
        acceptance_model=h1_base.acceptance_model,
    )
    _case = BlockchainCaseStudyService(seed=42, scenarios=100_000)
    result = _case.simulate_year1({"H1": f})["H1"]
    rows.append(
        (
            label,
            f"€{result.expected_eur:,.0f}",
            f"€{result.var_95_eur:,.0f}",
            f"€{result.p95_eur:,.0f}",
            f"€{result.std_eur:,.0f}",
        )
    )

In [ ]:
show.samples(
    headers=(
        "Uncertainty level",
        "Expected",
        "Business Value Floor",
        "P95 (ceiling)",
        "Std Dev",
    ),
    rows=rows,
    title="H1 — Effect of uncertainty on business value distribution",
    description="Same users, rate, and value per conversion — only uncertainty changes",
    footer="100,000 scenarios | seed 42",
)

### Live: annual growth rate — 3-year compounding for all features

In [ ]:
from fhs.application import BlockchainCaseStudyService

case = BlockchainCaseStudyService(seed=scenario.seed, scenarios=50_000)
year1 = case.simulate_year1(scenario.features_by_key)

rows = []
for f in scenario.features:
    key = next(k for k, feat in scenario.features_by_key.items() if feat.name == f.name)
    base = year1[key].expected_eur
    y2 = base * (1 + f.annual_growth_rate)
    y3 = y2 * (1 + f.annual_growth_rate)
    rows.append(
        (
            f.name,
            f"{f.annual_growth_rate:.0%}",
            f"€{base:,.0f}",
            f"€{y2:,.0f}",
            f"€{y3:,.0f}",
            f"€{base + y2 + y3:,.0f}",
        )
    )

In [ ]:
show.samples(
    headers=("Feature", "Growth Rate", "Year 1", "Year 2", "Year 3", "3yr Total"),
    rows=rows,
    title="Expected business value with annual growth (deterministic projection from simulated year 1)",
    footer="Year 2 and 3 are Year 1 × (1 + growth_rate)^n — uncertainty adds spread on top",
)

In [ ]:
rows = [
    (
        f.name,
        f"{f.expected_users:,}",
        f"{f.conversion_rate:.0%}",
        f"±{f.uncertainty:.0%}",
        f"€{f.business_value_per_conversion:,.2f}",
        f"€{f.development_cost:,.0f}",
        f"{f.installment_years}y",
        f"€{f.annual_operating_cost:,.0f}",
        f"{f.likelihood_of_non_delivery:.0%}",
    )
    for f in scenario.features
]

In [ ]:
show.samples(
    headers=(
        "Feature",
        "Users",
        "Conv. Rate",
        "Uncertainty",
        "Value/Conv",
        "Dev Cost",
        "Deprec.",
        "Oper. Cost/yr",
        "Non-Delivery Risk",
    ),
    rows=rows,
    title="Blockchain scenario — all feature parameters",
)

## 10 — JSON Schema validation

Every YAML file is validated against `scenario_schema.json` when loaded.
The cell below prints the required fields so you can see what is mandatory.

In [ ]:
import json
from pathlib import Path

schema_path = Path("scenario_schema.json")
schema = json.loads(schema_path.read_text())

print("=== Scenario-level required fields ===")
for f in schema.get("required", []):
    print(f"  {f}")

print("\n=== Feature-level required fields ===")
feature_def = schema["$defs"]["Feature"]
for f in feature_def.get("required", []):
    print(f"  {f}")

print("\n=== Feature-level optional fields with defaults ===")
for field_name, props in feature_def["properties"].items():
    if field_name not in feature_def.get("required", []) and "default" in props:
        print(f"  {field_name}: {props['default']}")

---

## Navigation

| File | Purpose |
|---|---|
| [blockchain.yaml](blockchain.yaml) | Blockchain case study scenario |
| [advanced-features.yaml](advanced-features.yaml) | 10-feature advanced portfolio scenario |
| [scenario_schema.json](scenario_schema.json) | JSON Schema — validation rules for all YAML files |
| [02-blockchain-case-study.ipynb](../02-blockchain-case-study.ipynb) | Edit scenario via interactive form |
| [03-capital-budgeting.ipynb](../03-blockchain-case-study-capital-budgeting.ipynb) | ROI, NPV, IRR, installment tables |